# 03 — Model Training & Evaluation
**Financial Risk Model | Training, Tuning, and Benchmarking**

This notebook covers:
1. Data loading and preprocessing
2. Class imbalance handling (SMOTE)
3. Cross-validation of all candidate models
4. Hyperparameter tuning (GridSearchCV)
5. Model comparison on test set
6. ROC, PR, and Calibration curves
7. Feature importances
8. Financial metrics: Gini, KS statistic, Gains table
9. Model selection and artifact export

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))
sys.path.insert(0, os.path.join('..', 'utils'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from data_loader import load_raw_data, generate_synthetic_dataset, split_data
from feature_engineering import engineer_features
from preprocessing import fit_preprocessor, transform, save_feature_names
from model_training import (
    get_model_catalogue, cross_validate_models, train_all_models,
    select_best_model, save_model
)
from model_evaluation import (
    compute_metrics, plot_confusion_matrix, plot_roc_curves,
    plot_precision_recall_curves, plot_feature_importance,
    plot_calibration_curve, full_classification_report
)
from metrics import gini_coefficient, ks_statistic, gains_table
from helpers import set_global_seed, timer

plt.style.use('seaborn-v0_8-whitegrid')
set_global_seed(42)
os.makedirs('../reports/figures', exist_ok=True)
os.makedirs('../models', exist_ok=True)
print('Environment ready ✅')

## 1. Load and Prepare Data

In [ ]:
RAW_PATH = '../data/raw/financial_risk_data.csv'

if not os.path.exists(RAW_PATH):
    df = generate_synthetic_dataset(n_samples=5000, save_path=RAW_PATH)
else:
    df = load_raw_data(filepath=RAW_PATH, config_path='../config/config.yaml')

df = engineer_features(df)

X_train, X_val, X_test, y_train, y_val, y_test = split_data(
    df, target_col='RiskFlag', test_size=0.20, val_size=0.10, random_state=42
)
print(f'Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}')
print(f'Train default rate: {y_train.mean()*100:.1f}%')

## 2. Preprocessing Pipeline

In [ ]:
with timer('Fit preprocessor'):
    preprocessor, feature_names = fit_preprocessor(
        X_train, config_path='../config/config.yaml',
        save_path='../models/preprocessor.pkl'
    )
    X_train_t = preprocessor.transform(X_train)
    X_val_t   = preprocessor.transform(X_val)
    X_test_t  = preprocessor.transform(X_test)

save_feature_names(feature_names, path='../models/feature_names.json')
print(f'Preprocessed shape: {X_train_t.shape}')
print(f'Total features after encoding: {len(feature_names)}')

## 3. Class Imbalance Analysis & SMOTE

In [ ]:
print('Class distribution before SMOTE:')
print(pd.Series(y_train).value_counts().to_string())

try:
    from imblearn.over_sampling import SMOTE
    smote = SMOTE(random_state=42, k_neighbors=5)
    X_train_sm, y_train_sm = smote.fit_resample(X_train_t, y_train)
    print('\nClass distribution after SMOTE:')
    print(pd.Series(y_train_sm).value_counts().to_string())
    USE_SMOTE = True
except ImportError:
    print('\n⚠️  imbalanced-learn not installed. Using class_weight="balanced" instead.')
    X_train_sm, y_train_sm = X_train_t, y_train
    USE_SMOTE = False

## 4. Cross-Validation Baseline

In [ ]:
import yaml
with open('../config/config.yaml') as f:
    cfg = yaml.safe_load(f)

catalogue = get_model_catalogue(cfg)
print('Models to evaluate:', list(catalogue.keys()))

with timer('Cross-validation'):
    cv_results = cross_validate_models(
        catalogue, X_train_sm, y_train_sm,
        cv_folds=5, scoring='roc_auc', random_state=42
    )

cv_df = pd.DataFrame(cv_results).T
cv_df.columns = ['CV Mean AUC', 'CV Std']
cv_df = cv_df.sort_values('CV Mean AUC', ascending=False)
print('\nCross-Validation Results:')
display(cv_df.style.background_gradient(cmap='Greens', subset=['CV Mean AUC']))

In [ ]:
# Visualise CV scores with error bars
fig, ax = plt.subplots(figsize=(9, 5))
models_list = list(cv_results.keys())
means = [cv_results[m]['mean'] for m in models_list]
stds  = [cv_results[m]['std']  for m in models_list]

bars = ax.bar(models_list, means, yerr=stds, capsize=8,
              color=['#2196F3', '#FF5722', '#4CAF50'],
              edgecolor='white', linewidth=1.5, error_kw={'linewidth': 2})
ax.axhline(0.5, color='red', linestyle='--', linewidth=1.5, label='Random baseline')
for bar, mean in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{mean:.4f}', ha='center', fontsize=11, fontweight='bold')
ax.set_ylabel('ROC-AUC (5-fold CV)', fontsize=12)
ax.set_title('Cross-Validation AUC — Model Comparison', fontsize=13, fontweight='bold')
ax.set_ylim([0.4, 1.0])
ax.legend()
plt.tight_layout()
plt.savefig('../reports/figures/cv_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Hyperparameter Tuning

In [ ]:
# ⚠️  Full grid search can take 5–15 minutes. 
# Set TUNE = False for a quick demo run.
TUNE = True

with timer('Model training + tuning'):
    trained_models = train_all_models(
        X_train_sm, y_train_sm,
        config_path='../config/config.yaml',
        tune=TUNE
    )

print('\nTrained models:', list(trained_models.keys()))

## 6. Validation Set Selection

In [ ]:
best_name, best_model = select_best_model(
    trained_models, X_val_t, y_val, scoring='roc_auc'
)
print(f'\n🏆 Best model: {best_name}')

## 7. Test Set Evaluation — All Models

In [ ]:
all_metrics = []
models_probs = {}

for name, estimator in trained_models.items():
    y_prob = estimator.predict_proba(X_test_t)[:, 1]
    y_pred = (y_prob >= 0.50).astype(int)
    models_probs[name] = y_prob
    metrics = compute_metrics(y_test.values, y_pred, y_prob, model_name=name)
    all_metrics.append(metrics)

results_df = pd.DataFrame(all_metrics).set_index('model')
print('\nTest Set Results:')
display(results_df[['accuracy','precision','recall','f1','roc_auc','gini','avg_precision']]
        .style.background_gradient(cmap='Greens', subset=['roc_auc','gini']))

## 8. Classification Reports

In [ ]:
for name, estimator in trained_models.items():
    y_pred = (models_probs[name] >= 0.50).astype(int)
    print(f'\n{'='*50}')
    print(f'  {name}')
    print(f'{'='*50}')
    print(full_classification_report(y_test.values, y_pred, model_name=name))

## 9. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, len(trained_models), figsize=(6 * len(trained_models), 5))
if len(trained_models) == 1:
    axes = [axes]

import seaborn as sns
from sklearn.metrics import confusion_matrix

for ax, (name, estimator) in zip(axes, trained_models.items()):
    y_pred = (models_probs[name] >= 0.50).astype(int)
    cm = confusion_matrix(y_test.values, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Low Risk', 'High Risk'],
                yticklabels=['Low Risk', 'High Risk'])
    ax.set_title(f'{name}\nConfusion Matrix', fontsize=11, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.savefig('../reports/figures/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. ROC & Precision-Recall Curves

In [ ]:
fig_roc = plot_roc_curves(
    models_probs, y_test.values,
    save_path='../reports/figures/roc_curves.png'
)
plt.show()

fig_pr = plot_precision_recall_curves(
    models_probs, y_test.values,
    save_path='../reports/figures/pr_curves.png'
)
plt.show()

## 11. Calibration Curves

In [ ]:
fig_cal = plot_calibration_curve(
    models_probs, y_test.values,
    save_path='../reports/figures/calibration_curves.png'
)
plt.show()
print('Well-calibrated models sit close to the diagonal.')
print('For IFRS 9 PD estimation, calibration is as important as discrimination.')

## 12. Feature Importances

In [ ]:
for name, estimator in trained_models.items():
    fig = plot_feature_importance(
        estimator, feature_names, top_n=20, model_name=name,
        save_path=f'../reports/figures/feature_importance_{name.lower()}.png'
    )
    if fig:
        plt.show()
        plt.close()

## 13. Advanced Financial Metrics

In [ ]:
print('Advanced Credit-Risk Metrics for Best Model:')
print(f'  Model          : {best_name}')

best_probs = models_probs[best_name]
gini = gini_coefficient(y_test.values, best_probs)
ks   = ks_statistic(y_test.values, best_probs)

print(f'  Gini Coefficient: {gini:.4f}  (benchmark: ≥0.30 acceptable, ≥0.50 strong)')
print(f'  KS Statistic    : {ks:.4f}  (benchmark: ≥0.30 acceptable)')

# Interpretation
if gini >= 0.50:
    print('  → Gini: STRONG discriminatory power ✅')
elif gini >= 0.30:
    print('  → Gini: ACCEPTABLE discriminatory power ⚠️')
else:
    print('  → Gini: WEAK — model needs improvement ❌')

In [ ]:
# Gains table
gt = gains_table(y_test.values, best_probs, n_deciles=10)
print(f'\nGains Table — {best_name}:')
print(gt.to_string(index=False))

# Visualise cumulative gains
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Cumulative gains curve
axes[0].plot(gt['decile'], gt['pct_defaults'], 'o-', color='#2196F3',
             linewidth=2.5, markersize=7, label=f'{best_name}')
axes[0].plot([1, 10], [10, 100], 'k--', linewidth=1.5, label='Random model')
axes[0].set_xlabel('Decile (1 = Highest Risk)', fontsize=12)
axes[0].set_ylabel('% Defaults Captured', fontsize=12)
axes[0].set_title('Cumulative Gains Curve', fontsize=13, fontweight='bold')
axes[0].legend()

# Lift chart
axes[1].bar(gt['decile'], gt['lift'], color=plt.cm.RdYlGn_r(gt['lift'] / gt['lift'].max()),
            edgecolor='white')
axes[1].axhline(1, color='red', linestyle='--', linewidth=1.5, label='No-lift baseline')
axes[1].set_xlabel('Decile', fontsize=12)
axes[1].set_ylabel('Lift', fontsize=12)
axes[1].set_title('Lift Chart', fontsize=13, fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('../reports/figures/gains_lift.png', dpi=150, bbox_inches='tight')
plt.show()

top2_decile_capture = gt[gt['decile'] <= 2]['pct_defaults'].max()
print(f'\nTop 20% of risk scores capture {top2_decile_capture:.1f}% of all defaults.')

## 14. Threshold Sensitivity Analysis
Explore how precision, recall and F1 change with the classification threshold.

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = np.arange(0.1, 0.9, 0.05)
precisions, recalls, f1s = [], [], []

for thresh in thresholds:
    y_pred_t = (best_probs >= thresh).astype(int)
    precisions.append(precision_score(y_test.values, y_pred_t, zero_division=0))
    recalls.append(recall_score(y_test.values, y_pred_t, zero_division=0))
    f1s.append(f1_score(y_test.values, y_pred_t, zero_division=0))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thresholds, precisions, 'b-o', markersize=5, label='Precision', linewidth=2)
ax.plot(thresholds, recalls,   'r-o', markersize=5, label='Recall',    linewidth=2)
ax.plot(thresholds, f1s,       'g-o', markersize=5, label='F1',        linewidth=2)
ax.axvline(0.50, color='grey', linestyle='--', linewidth=1.5, label='Default threshold (0.5)')

best_f1_thresh = thresholds[np.argmax(f1s)]
ax.axvline(best_f1_thresh, color='green', linestyle=':', linewidth=2,
           label=f'Best F1 threshold ({best_f1_thresh:.2f})')

ax.set_xlabel('Classification Threshold', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title(f'{best_name} — Threshold Sensitivity Analysis', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('../reports/figures/threshold_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Best F1 threshold  : {best_f1_thresh:.2f}')
print(f'F1 @ 0.50          : {f1s[list(thresholds).index(0.50) if 0.50 in thresholds else len(thresholds)//2]:.4f}')
print(f'F1 @ best threshold: {max(f1s):.4f}')

## 15. Save Models

In [ ]:
for name, estimator in trained_models.items():
    save_model(estimator, name, model_dir='../models')

# Save best model under canonical alias
save_model(best_model, best_name, model_dir='../models', filename='risk_model.pkl')

# Persist evaluation results
results_df.to_csv('../models/evaluation_results.csv')

print('\n✅ All artefacts saved:')
for f in os.listdir('../models'):
    size = os.path.getsize(f'../models/{f}') / 1024
    print(f'   {f:40s}  {size:.1f} KB')

print(f'\n🏆 Best Model : {best_name}')
print(f'   ROC-AUC    : {results_df.loc[best_name, "roc_auc"]:.4f}')
print(f'   Gini       : {gini:.4f}')
print(f'   KS         : {ks:.4f}')